# Colab Demo: Supply Chain Chatbot with Gemini Integration

This notebook demonstrates the supply chain optimization chatbot with Gemini-powered explanations. The system supports progressive inquiry for classroom use, extracting entities from natural language, constructing mathematical models, and providing AI-enhanced explanations.

## Features Demonstrated
- Natural language problem formulation
- Mathematical model generation with Pyomo
- Gemini AI explanations (with fallback)
- What-if scenario analysis
- Theorem applicability checking

In [ ]:
# Install dependencies
!pip install -q google-generativeai pydantic pyomo pandas numpy networkx matplotlib

In [ ]:
# Set GEMINI_API_KEY
import os
from getpass import getpass

if "GEMINI_API_KEY" not in os.environ:
    os.environ["GEMINI_API_KEY"] = getpass("Enter your GEMINI_API_KEY: ")

print("GEMINI_API_KEY configured:", "GEMINI_API_KEY" in os.environ)

In [ ]:
# Clone repository and install requirements
import os

repo_url = "https://github.com/your-org/ChatbotLP.git"  # Replace with actual repo URL
repo_dir = "ChatbotLP"

if not os.path.isdir(repo_dir):
    !git clone {repo_url} {repo_dir}

os.chdir(repo_dir)
!pip install -q -r requirements.txt

In [ ]:
# Imports
from src.chatbot_engine import run_chatbot_session
from src.state_manager import StateManager
from src.gemini_explanation_provider import GeminiExplanationProvider
from src.llm_adapter import LLMProviderRegistry, RuleBasedProvider
from src.chatbot_engine import IntentRouter
from src.parser import parse_supply_chain_text
from src.response_generator import generate_response
from src.schema import ProblemState, Node, Product, Supplier, Consumer, Bid

In [ ]:
# Create example problem
mgr = StateManager()
state = mgr.get_state()
state.problem_title = "Simple Supply Chain Example"

# Add nodes
state.add_node(Node(id="farm", name="Farm"))
state.add_node(Node(id="market", name="Market"))

# Add product
state.add_product(Product(id="wheat", name="Wheat"))

# Add supplier and consumer
state.add_supplier(Supplier(id="supplier1", node="farm", product="wheat", capacity=100))
state.add_consumer(Consumer(id="consumer1", node="market", product="wheat", capacity=80))

# Add bid
state.add_bid(Bid(id="bid1", owner_id="supplier1", owner_type="supplier", product_id="wheat", price=10.0, quantity=100))

print("Problem state created with:")
print(f"  Nodes: {[n.id for n in state.nodes]}")
print(f"  Products: {[p.id for p in state.products]}")
print(f"  Suppliers: {[s.id for s in state.suppliers]}")
print(f"  Consumers: {[c.id for c in state.consumers]}")
print(f"  Bids: {[b.id for b in state.bids]}")

In [ ]:
# Instantiate GeminiExplanationProvider
try:
    gemini_provider = GeminiExplanationProvider()
    print("GeminiExplanationProvider instantiated successfully")
except Exception as e:
    print(f"Failed to instantiate GeminiExplanationProvider: {e}")
    gemini_provider = None

In [ ]:
# Register provider
if gemini_provider:
    class GeminiLLMProvider(RuleBasedProvider):
        def get_explanation_generator(self):
            return gemini_provider

    registry = LLMProviderRegistry.get_instance()
    registry.set_provider(
        GeminiLLMProvider(
            intent_router=IntentRouter(),
            parse_function=parse_supply_chain_text,
            generate_function=generate_response,
        )
    )
    print("Gemini provider registered successfully")
else:
    print("Using default rule-based provider")

In [ ]:
# Run chatbot session with use_llm=True
result = run_chatbot_session(state, "Explain the current supply chain problem", use_llm=True)

print("Chatbot Session Result:")
print(f"Intent: {result['intent']}")
print(f"Success: {result['success']}")
print("\nResponse:")
print(result['response'])

In [ ]:
# Demonstrate fallback behavior
import os

# Temporarily remove API key
original_key = os.environ.pop("GEMINI_API_KEY", None)
LLMProviderRegistry.get_instance().reset()

print("API key removed for fallback test")

# Run session without LLM
fallback_result = run_chatbot_session(state, "Explain the current supply chain problem", use_llm=True)

print("\nFallback Response (rule-based):")
print(fallback_result['response'])

# Restore key
if original_key:
    os.environ["GEMINI_API_KEY"] = original_key
    print("\nAPI key restored")

# Conclusion

This notebook demonstrated:
1. Setting up the supply chain chatbot environment
2. Configuring Gemini AI for enhanced explanations
3. Creating and solving supply chain optimization problems
4. Graceful fallback to rule-based explanations when AI is unavailable

The system maintains mathematical correctness through Pyomo while providing rich, AI-generated explanations suitable for educational use.